In [4]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split

!pip install keras_tuner # Install keras_tuner
import keras_tuner as kt

import re
import gc

print("TensorFlow version:", tf.__version__)
print("Keras Tuner imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 8.3 MB/s eta 0:00:00
TensorFlow version: 2.19.0
Keras Tuner imported successfully


In [6]:
print("Loading dataset in chunks...")
chunk_size = 5000
chunks = []

for chunk in pd.read_csv('random_evals.csv', chunksize=chunk_size):
    chunks.append(chunk)
    if len(chunks) * chunk_size >= 50000:  # Load 50k rows max
        break

df = pd.concat(chunks, ignore_index=True)
print(f"Loaded {len(df)} rows")
print(df.head())


Loading dataset in chunks...
Loaded 50000 rows
                                                 FEN Evaluation
0  rnbqkb1r/pppppppp/B4n2/8/4P3/8/PPPP1PPP/RNBQK1...       -459
1  rnbqkb1r/pppppppp/5n2/1B6/4P3/8/PPPP1PPP/RNBQK...       -125
2  rnbqkbnr/p1pppppp/8/1p6/4P3/8/PPPP1PPP/RNBQKBN...       +198
3  rnbqkb1r/pppppppp/5n2/8/4P3/7N/PPPP1PPP/RNBQKB...       -155
4  rnbqkbnr/ppppp1pp/8/5p2/4P3/8/PPPP1PPP/RNBQKBN...       +209


In [7]:

# ===== CELL 3: PROCESS EVALUATION SCORES =====
def process_eval(eval_str):
    """Convert evaluation string to numeric value"""
    eval_str = str(eval_str).strip()

    # Handle mate scores
    if '#' in eval_str:
        match = re.search(r'#([+-]?\d+)', eval_str)
        if match:
            moves_to_mate = int(match.group(1))
            return 10000.0 if moves_to_mate > 0 else -10000.0

    # Handle regular scores
    try:
        return float(eval_str.replace('+', ''))
    except:
        return 0.0

print("\nProcessing evaluation scores...")
df['eval_numeric'] = df['Evaluation'].apply(process_eval)
print(f"Eval range: {df['eval_numeric'].min()} to {df['eval_numeric'].max()}")
print(f"Eval mean: {df['eval_numeric'].mean():.2f}, std: {df['eval_numeric'].std():.2f}")



Processing evaluation scores...
Eval range: -10000.0 to 10000.0
Eval mean: 200.31, std: 2501.56


In [8]:

# ===== CELL 4: BITBOARD + FEATURES CONVERSION =====
def fen_to_bitboards_and_features(fen):
    """
    Convert FEN to bitboard representation + extra features
    Returns: array of shape (781,)
    - 768: 12 bitboards (12 piece types × 64 squares)
    - 4: castling rights
    - 8: en passant file (one-hot)
    - 1: side to move
    """
    parts = fen.split()
    board_fen = parts[0]
    side_to_move = parts[1]
    castling = parts[2]
    en_passant = parts[3]

    # ===== BITBOARDS (768) =====
    piece_types = ['P', 'p', 'N', 'n', 'B', 'b', 'R', 'r', 'Q', 'q', 'K', 'k']
    bitboards = np.zeros(768, dtype=np.float32)

    # Parse board and fill bitboards
    square = 0
    for rank in board_fen.split('/'):
        for char in rank:
            if char.isdigit():
                square += int(char)
            else:
                piece_idx = piece_types.index(char)
                bitboards[piece_idx * 64 + square] = 1.0
                square += 1

    # ===== CASTLING RIGHTS (4) =====
    castling_features = np.zeros(4, dtype=np.float32)
    if 'K' in castling:
        castling_features[0] = 1.0
    if 'Q' in castling:
        castling_features[1] = 1.0
    if 'k' in castling:
        castling_features[2] = 1.0
    if 'q' in castling:
        castling_features[3] = 1.0

    # ===== EN PASSANT (8) =====
    en_passant_features = np.zeros(8, dtype=np.float32)
    if en_passant != '-':
        file = ord(en_passant[0]) - ord('a')
        en_passant_features[file] = 1.0

    # ===== SIDE TO MOVE (1) =====
    side_feature = np.array([1.0 if side_to_move == 'w' else 0.0], dtype=np.float32)

    # Concatenate all
    vector = np.concatenate([bitboards, castling_features, en_passant_features, side_feature])
    return vector

print("Converting FEN to bitboards + features...")
print("This may take a minute...")

# Process in chunks to avoid memory issues
X_list = []
for i, fen in enumerate(df['FEN']):
    if i % 10000 == 0:
        print(f"  Processed {i}/{len(df)}")
    X_list.append(fen_to_bitboards_and_features(fen))

X = np.array(X_list, dtype=np.float32)
y = df['eval_numeric'].values.astype(np.float32)

# Normalize evaluation scores
y = np.clip(y / 5.0, -1, 1)

print(f"Input shape: {X.shape}")
print(f"Output shape: {y.shape}")
print(f"Input feature count: {X.shape[1]}")

# Clear memory
del chunks, df, X_list
gc.collect()

Converting FEN to bitboards + features...
This may take a minute...
  Processed 0/50000
  Processed 10000/50000
  Processed 20000/50000
  Processed 30000/50000
  Processed 40000/50000
Input shape: (50000, 781)
Output shape: (50000,)
Input feature count: 781


124

In [9]:

# ===== CELL 5: TRAIN TEST SPLIT =====
print("\nSplitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train set: {X_train.shape}")
print(f"Test set: {X_test.shape}")



Splitting data...
Train set: (40000, 781)
Test set: (10000, 781)


In [10]:

# ===== CELL 6: BUILD MODEL BUILDER FOR HYPERPARAMETER TUNING =====
def build_model(hp):
    """
    Model builder for Keras Tuner
    Hyperparameters to tune:
    - Number of hidden layers
    - Units per layer
    - Dropout rate
    - Learning rate
    """
    model = keras.Sequential()

    # Input layer
    model.add(keras.layers.Input(shape=(781,)))

    # Hidden layers (tune number of layers: 2-4)
    num_layers = hp.Int('num_layers', min_value=2, max_value=4, step=1)
    for i in range(num_layers):
        units = hp.Int(f'units_{i}', min_value=128, max_value=512, step=64)
        model.add(keras.layers.Dense(units, activation='relu'))

        dropout_rate = hp.Float(f'dropout_{i}', min_value=0.1, max_value=0.5, step=0.1)
        model.add(keras.layers.Dropout(dropout_rate))

    # Output layer
    model.add(keras.layers.Dense(1, activation='linear'))

    # Compile
    learning_rate = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )

    return model

print("Model builder created successfully")


Model builder created successfully


In [11]:

# ===== CELL 7: SETUP KERAS TUNER =====
print("Setting up Keras Tuner...")

tuner = kt.Hyperband(
    build_model,
    objective='val_loss',
    max_epochs=10,  # Limit epochs during tuning
    factor=3,
    directory='chess_tuner',
    project_name='chess_eval_tuning',
    overwrite=True
)

print("Tuner created successfully")
print("\nTuner search space:")
print("- Number of layers: 2-4")
print("- Units per layer: 128-512 (step 64)")
print("- Dropout rate: 0.1-0.5 (step 0.1)")
print("- Learning rate: 1e-4 to 1e-2 (log scale)")


Setting up Keras Tuner...
Tuner created successfully

Tuner search space:
- Number of layers: 2-4
- Units per layer: 128-512 (step 64)
- Dropout rate: 0.1-0.5 (step 0.1)
- Learning rate: 1e-4 to 1e-2 (log scale)


In [12]:

# ===== CELL 8: READY FOR TRAINING =====
print("\n" + "="*50)
print("SETUP COMPLETE - READY FOR HYPERPARAMETER TUNING")
print("="*50)
print(f"\nTraining data shape: {X_train.shape}")
print(f"Test data shape: {X_test.shape}")
print(f"\nNext step: Run tuner.search()")
print("Command: tuner.search(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)")
print("\nThen: best_model = tuner.get_best_models(num_models=1)[0]")


SETUP COMPLETE - READY FOR HYPERPARAMETER TUNING

Training data shape: (40000, 781)
Test data shape: (10000, 781)

Next step: Run tuner.search()
Command: tuner.search(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)

Then: best_model = tuner.get_best_models(num_models=1)[0]


In [13]:
tuner.search(X_train, y_train, epochs=10, validation_split=0.1, verbose=1)
best_model = tuner.get_best_models(num_models=1)[0]

Trial 30 Complete [00h 01m 39s]
val_loss: 0.17971643805503845

Best val_loss So Far: 0.17411358654499054
Total elapsed time: 00h 26m 40s


In [16]:
best_hp = tuner.get_best_hyperparameters(num_trials=1)[0]
print(best_hp.values)


{'num_layers': 2, 'units_0': 192, 'dropout_0': 0.2, 'units_1': 512, 'dropout_1': 0.1, 'learning_rate': 0.0006948967296604217, 'units_2': 384, 'dropout_2': 0.2, 'units_3': 384, 'dropout_3': 0.4, 'tuner/epochs': 10, 'tuner/initial_epoch': 4, 'tuner/bracket': 2, 'tuner/round': 2, 'tuner/trial_id': '0012'}
